# Ensemble Forecasting

Notebook para clonar el repo, montar Google Drive, cargar predicciones cacheadas y ensamblar los modelos.

`MOMENT` y `Moirai` siguen la misma idea del main anterior: primero se buscan en Drive; este notebook no los reentrena.

In [ ]:
import os
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
REPO_DIR = '/content/heartrate-forecasting' if IN_COLAB else os.getcwd()

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        subprocess.check_call([
            'git',
            'clone',
            'https://github.com/allandbb/heartrate-forecasting.git',
            REPO_DIR,
        ])
    else:
        subprocess.check_call(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
    os.chdir(REPO_DIR)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        'git+https://github.com/moment-timeseries-foundation-model/moment.git',
    ])
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        'keras-tcn',
        'tensorflow',
    ])
    print('Si acabas de instalar dependencias por primera vez, reinicia el runtime y sigue desde la siguiente celda.')
else:
    print(f'Usando repo local en: {REPO_DIR}')

In [ ]:
import importlib
import os
import sys

if IN_COLAB:
    from google.colab import drive

    drive.mount('/content/drive')
    os.chdir(REPO_DIR)
    CACHE_DIR = '/content/drive/MyDrive/hr_forecasting_cache'
else:
    CACHE_DIR = 'cache'

os.makedirs(CACHE_DIR, exist_ok=True)

repo_main_path = os.path.join(REPO_DIR, 'main.py')
if not os.path.exists(repo_main_path):
    raise FileNotFoundError(
        f'No se encontro {repo_main_path}. Ejecuta de nuevo la celda de clonacion/sync y verifica que el commit con main.py este en GitHub.'
    )

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import main
importlib.reload(main)

print(f'Repo: {os.getcwd()}')
print(f'Cache: {CACHE_DIR}')

In [ ]:
import json
import os

import numpy as np
import pandas as pd

MODELS = list(main.DEFAULT_MODELS)
# Si quieres incluir modelos foundation que requieren ajuste adicional:
# MODELS = list(main.DEFAULT_MODELS) + list(main.FOUNDATION_MODELS)
DATASET_DIR = 'dataset'
INPUT_SIZE = 200
OUTPUT_SIZE = 200
VALIDATION_SIZE = 0.1
RANDOM_STATE = 42
SELECT_BY = 'MAPE'  # compatibilidad legacy; ya no decide el ensemble final sobre eval
OBJECTIVE_METRIC = 'MAPE'
ENSEMBLE_ACTIVE_METHOD = 'optimize'
FORCE_RECOMPUTE = False

LEGACY_DRIVE_CACHE = {
    'moment': {
        'eval': 'y_pred_moment.npy',
        'fit': 'y_pred_val_moment.npy',
        'metrics': 'metrics_moment.json',
    },
    'moirai': {
        'eval': 'y_pred_moirai.npy',
        'fit': 'y_pred_val_moirai.npy',
        'metrics': 'metrics_moirai.json',
    },
    'tcn': {
        'eval': 'y_pred_tcn.npy',
        'fit': 'y_pred_val_tcn.npy',
        'metrics': 'metrics_tcn.json',
    },
    'nbeats': {
        'eval': 'y_pred_nbeats.npy',
        'fit': 'y_pred_val_nbeats.npy',
        'metrics': 'metrics_nbeats.json',
    },
}


def load_bundle_from_paths(paths):
    if not all(os.path.exists(paths[key]) for key in ['eval', 'fit', 'metrics']):
        return None
    with open(paths['metrics'], 'r', encoding='utf-8') as f:
        metrics = json.load(f)
    return {
        'eval': np.load(paths['eval']),
        'fit': np.load(paths['fit']),
        'metrics': metrics,
    }


def load_cached_bundle(model_key):
    structured_paths = main.prediction_cache_paths(CACHE_DIR, model_key)
    structured = main.load_cached_predictions(structured_paths)
    if structured is not None:
        print(f'Usando cache estructurado para {model_key}.')
        return structured

    legacy_spec = LEGACY_DRIVE_CACHE.get(model_key)
    if legacy_spec is None:
        return None

    legacy_paths = {
        key: os.path.join(CACHE_DIR, value)
        for key, value in legacy_spec.items()
    }
    legacy = load_bundle_from_paths(legacy_paths)
    if legacy is None:
        return None

    print(f'Usando cache legado de Drive para {model_key}.')
    main.save_cached_predictions(structured_paths, legacy['eval'], legacy['fit'], legacy['metrics'])
    return legacy


def load_or_predict_for_ensemble(model_key, spec, datasets):
    if not FORCE_RECOMPUTE:
        cached = load_cached_bundle(model_key)
        if cached is not None:
            return cached

    if spec.get('needs_fit', False):
        raise RuntimeError(
            f"{spec['label']} necesita predicciones cacheadas en Drive o una corrida previa; este notebook no entrena ese modelo."
        )

    return main.run_model(
        model_key=model_key,
        spec=spec,
        datasets=datasets,
        cache_dir=CACHE_DIR,
        force_recompute=FORCE_RECOMPUTE,
    )

In [ ]:
main.ensure_dir(CACHE_DIR)
datasets = main.prepare_datasets(
    dataset_dir=DATASET_DIR,
    cache_dir=CACHE_DIR,
    input_size=INPUT_SIZE,
    output_size=OUTPUT_SIZE,
    validation_size=VALIDATION_SIZE,
    random_state=RANDOM_STATE,
)

In [ ]:
registry = main.build_model_registry()
individual_results = {}
eval_predictions = {}
fit_predictions = {}

for model_key in MODELS:
    spec = registry[model_key]
    label = spec['label']
    try:
        bundle = load_or_predict_for_ensemble(model_key, spec, datasets)
    except Exception as exc:
        print(f'[SKIP] {label}: {exc}')
        continue

    eval_predictions[label] = bundle['eval']
    fit_predictions[label] = bundle['fit']

if not eval_predictions:
    raise RuntimeError('No se pudo cargar ningun modelo para el ensemble.')

In [ ]:
original_scale = main.align_and_restore_original_scale(datasets, eval_predictions, fit_predictions)
datasets['y_test'] = original_scale['y_true_eval']
datasets['y_va'] = original_scale['y_true_fit']
datasets['ids_test'] = original_scale['ids_eval']
datasets['ids_va'] = original_scale['ids_fit']
eval_predictions = original_scale['eval_predictions']
fit_predictions = original_scale['fit_predictions']

individual_results = main.compute_individual_results(datasets['y_test'], eval_predictions)

df_individual = pd.DataFrame(individual_results).T
df_individual.index.name = 'Modelo'
if 'MAPE' in df_individual.columns:
    df_individual = df_individual.sort_values('MAPE')
df_individual

In [ ]:
ensemble_results = {}

if len(eval_predictions) >= 2:
    ens = main.EnsembleWrapper()
    for label, pred in eval_predictions.items():
        ens.add_model(label, pred, split='eval')
    for label, pred in fit_predictions.items():
        ens.add_model(label, pred, split='fit')

    ensemble_results = ens.compare_methods(
        y_true_eval=datasets['y_test'],
        y_true_fit=datasets['y_va'],
        select_by=SELECT_BY,
        objective_metric=OBJECTIVE_METRIC,
        active_method=ENSEMBLE_ACTIVE_METHOD,
    )
else:
    print('No hay suficientes modelos para construir ensemble. Se requieren al menos 2.')

df_ensemble = pd.DataFrame(ensemble_results).T if ensemble_results else pd.DataFrame()
if not df_ensemble.empty:
    df_ensemble.index.name = 'Metodo'
df_ensemble

In [ ]:
main.save_final_tables(CACHE_DIR, individual_results, ensemble_results)

final_results = {}
final_results.update(individual_results)
final_results.update(ensemble_results)

df_final = pd.DataFrame(final_results).T
df_final.index.name = 'Modelo / Metodo'
if 'MAPE' in df_final.columns:
    df_final = df_final.sort_values('MAPE')
df_final